In [1]:
# ==========================================
# Step 1: Import Libraries
# ==========================================
import sys
import spacy
import pprint
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

In [2]:
# ==========================================
# Step 2: Print Environment Info
# ==========================================
print("Python:", sys.executable)
print("spaCy:", spacy.__version__)

# ==========================================
# Step 3: Load Dataset
# ==========================================
# Ensure your CSV has columns: 'id' and 'text'
df = pd.read_csv("E://CV//Internship//Coding_Challenge_Omkar_Pawar//data//cleaned_sentiment_dataset.csv")
print("Data loaded successfully! Shape:", df.shape)
print(df.head())

# Drop rows with missing text
df = df.dropna(subset=['text']).reset_index(drop=True)
df

Python: C:\Users\Hp\PythonProjects\project_1\venv\Scripts\python.exe
spaCy: 3.8.7
Data loaded successfully! Shape: (70427, 4)
     id      product sentiment  \
0  2401  Borderlands  Positive   
1  2401  Borderlands  Positive   
2  2401  Borderlands  Positive   
3  2401  Borderlands  Positive   
4  2401  Borderlands  Positive   

                                                text  
0  im getting on borderlands and i will murder yo...  
1  I am coming to the borders and I will kill you...  
2  im getting on borderlands and i will kill you ...  
3  im coming on borderlands and i will murder you...  
4  im getting on borderlands 2 and i will murder ...  


,id,product,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
...,...,...,...,...
70422,7516,LeagueOfLegends,Neutral,♥️ Suikoden 2\r\n1️⃣ Alex Kidd in Miracle Worl...
70423,5708,HomeDepot,Positive,Thank you to Matching funds Home Depot RW paym...
70424,2165,CallOfDuty,Neutral,Late night stream with the boys! Come watch so...
70425,4891,GrandTheftAuto(GTA),Irrelevant,⭐️ Toronto is the arts and culture capital of ...


In [3]:
# ==========================================
# Step 4: Load Embedding Model
# ==========================================
model_name = "all-mpnet-base-v2"
print(f"Loading model: {model_name}")
model = SentenceTransformer(model_name, device='cpu')

Loading model: all-mpnet-base-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Hp\PythonProjects\project_1\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hp\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
# ==========================================
# Step 5: Generate Embeddings
# ==========================================
texts = df["text"].astype(str).tolist()

print("Generating embeddings... (this may take a few minutes)")
embeddings = model.encode(
    texts,
    batch_size=32,                # safe for most CPUs (adjust if RAM ≥16GB → 64)
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True     # ensures consistent cosine-space embeddings
)

# Optional manual normalization (for older versions)
embeddings = normalize(embeddings)

print("Embeddings generated successfully!")
print("Shape of embeddings:", embeddings.shape)

Generating embeddings... (this may take a few minutes)


Batches:   0%|          | 0/2201 [00:00<?, ?it/s]

Embeddings generated successfully!
Shape of embeddings: (70427, 768)


In [12]:
# ==========================================
# Step 6: Combine IDs, Texts, and Embeddings
# ==========================================

# Create a new DataFrame containing id, text, and embedding
emb_df = pd.DataFrame({
    "id": df["id"],
    "text": df["text"],
    "product": df['product'],
    "sentiment": df['sentiment'],
    "embedding": [emb.tolist() for emb in embeddings]
})

In [13]:
emb_df

,id,text,product,sentiment,embedding
0,2401,im getting on borderlands and i will murder yo...,Borderlands,Positive,"[-0.004616821184754372, 0.06128309667110443, -..."
1,2401,I am coming to the borders and I will kill you...,Borderlands,Positive,"[-0.006876182742416859, 0.1059049740433693, -0..."
2,2401,im getting on borderlands and i will kill you ...,Borderlands,Positive,"[-0.003253370290622115, 0.04449448734521866, -..."
3,2401,im coming on borderlands and i will murder you...,Borderlands,Positive,"[0.027048178017139435, 0.08019574731588364, -0..."
4,2401,im getting on borderlands 2 and i will murder ...,Borderlands,Positive,"[0.0034572561271488667, 0.07546985894441605, -..."
...,...,...,...,...,...
70422,7516,♥️ Suikoden 2\r\n1️⃣ Alex Kidd in Miracle Worl...,LeagueOfLegends,Neutral,"[-0.039317622780799866, -0.0004228319739922881..."
70423,5708,Thank you to Matching funds Home Depot RW paym...,HomeDepot,Positive,"[-0.02414911426603794, 0.05882412940263748, -0..."
70424,2165,Late night stream with the boys! Come watch so...,CallOfDuty,Neutral,"[-0.06478378921747208, 0.0021919060964137316, ..."
70425,4891,⭐️ Toronto is the arts and culture capital of ...,GrandTheftAuto(GTA),Irrelevant,"[-0.03524599224328995, 0.05923598259687424, -0..."


In [14]:
# ==========================================
# Step 7: Save Embeddings
# ==========================================
# Option 1: Save as compressed NumPy file
np.savez_compressed("E://CV//Internship//Coding_Challenge_Omkar_Pawar//notebooks//embeddings_out//text_embeddings_mpnet.npz", ids=df["id"].values, embeddings=embeddings)
